# EDA — Credit Card Churn

Exploration des données clients : KPIs churn, distributions et corrélations.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from churn.config import CHURN_POSITIVE_LABEL, RAW_DATA_PATH, TARGET_COLUMN
from churn.ingest import load_raw_csv

sns.set_theme(style="whitegrid")
df = load_raw_csv()

## Aperçu des données

In [ ]:
print(f"Lignes: {len(df):,} | Colonnes: {df.shape[1]}")
df.head()

In [ ]:
df.info()
df.isnull().sum().sort_values(ascending=False).head(10)

## KPI principal — taux de churn

In [ ]:
churn_counts = df[TARGET_COLUMN].value_counts()
churn_rate = (df[TARGET_COLUMN] == CHURN_POSITIVE_LABEL).mean() * 100

print(churn_counts)
print(f"\nTaux de churn: {churn_rate:.2f}%")

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x=TARGET_COLUMN, ax=ax, palette="Set2")
ax.set_title("Répartition Existing vs Attrited Customer")
plt.tight_layout()
plt.show()

## Variables numériques clés

In [ ]:
numeric_cols = [
    "Customer_Age",
    "Credit_Limit",
    "Total_Trans_Amt",
    "Total_Trans_Ct",
    "Months_Inactive_12_mon",
    "Contacts_Count_12_mon",
]

df[numeric_cols].describe().T

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flat, numeric_cols):
    sns.histplot(data=df, x=col, hue=TARGET_COLUMN, kde=True, ax=ax, element="step")
    ax.set_title(col)
plt.tight_layout()
plt.show()

## Variables catégorielles vs churn

In [ ]:
cat_cols = ["Gender", "Education_Level", "Marital_Status", "Income_Category", "Card_Category"]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.flat, cat_cols):
    rate = df.groupby(col)[TARGET_COLUMN].apply(lambda s: (s == CHURN_POSITIVE_LABEL).mean())
    rate.sort_values(ascending=False).plot(kind="bar", ax=ax, color="steelblue")
    ax.set_title(f"Taux de churn par {col}")
    ax.set_ylabel("Churn rate")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## Corrélations (features numériques)

In [ ]:
df_eda = df.copy()
df_eda["is_churn"] = (df_eda[TARGET_COLUMN] == CHURN_POSITIVE_LABEL).astype(int)

corr_cols = numeric_cols + ["is_churn"]
corr = df_eda[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matrice de corrélation")
plt.tight_layout()
plt.show()

print("Corrélations avec is_churn:")
print(corr["is_churn"].sort_values(ascending=False))

## Insights EDA (synthèse)

- Le taux de churn est d'environ **16 %** (classe minoritaire → SMOTE recommandé).
- **Total_Trans_Amt**, **Total_Trans_Ct** et **Contacts_Count_12_mon** sont des signaux forts.
- **Months_Inactive_12_mon** élevé = risque de churn accru.
- Segments **Blue** et clients peu actifs à cibler en priorité marketing.